# Network Topology and Spiking Dynamics

**SC-NeuroCore v3.14** — How network structure shapes emergent activity.

SC-NeuroCore provides 6 connectivity generators. This notebook builds
identical LIF populations with different topologies and compares
their spiking behaviour under the same Poisson drive.

1. **Random** (Erdos-Renyi)
2. **Small-world** (Watts-Strogatz)
3. **Scale-free** (Barabasi-Albert)
4. **Ring** (nearest-neighbour)
5. **Grid** (2D lattice)
6. **All-to-all** (full connectivity)

> © 1998–2026 Miroslav Šotek. All rights reserved.  
> License: GNU AFFERO GENERAL PUBLIC LICENSE v3 | Commercial Licensing Available  
> Contact: www.anulum.li | protoscience@anulum.li

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sc_neurocore.network.topology import (
    random_connectivity,
    small_world,
    scale_free,
    ring_topology,
    grid_topology,
    all_to_all,
)
from sc_neurocore.network.population import Population
from sc_neurocore.network.projection import Projection
from sc_neurocore.network.network import Network
from sc_neurocore.network.monitor import SpikeMonitor
from sc_neurocore.network.stimulus import PoissonInput

print("SC-NeuroCore topology demo")

## 1. Connectivity Matrices

Each generator returns CSR arrays `(indptr, indices, data)` for
sparse connectivity. Below we visualise the adjacency patterns
for N=100 neurons.

In [ ]:
N = 100
W = 0.05

topologies = {
    "Random (p=0.1)": random_connectivity(N, N, p=0.1, weight=W, seed=42),
    "Small-world (k=8, p=0.1)": small_world(N, k=8, p_rewire=0.1, weight=W, seed=42),
    "Scale-free (m=3)": scale_free(N, m=3, weight=W, seed=42),
    "Ring (k=4)": ring_topology(N, k=4, weight=W),
    "Grid (10×10, r=1)": grid_topology(10, 10, radius=1, weight=W),
    "All-to-all": all_to_all(N, N, weight=W / 10),  # scale down to avoid saturation
}


def csr_to_dense(indptr, indices, data, n):
    """Convert CSR to dense matrix for visualisation."""
    mat = np.zeros((n, n))
    for i in range(n):
        for k in range(indptr[i], indptr[i + 1]):
            j = indices[k]
            if j < n:
                mat[i, j] = data[k]
    return mat


fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for ax, (name, (indptr, indices, data)) in zip(axes.flat, topologies.items()):
    mat = csr_to_dense(indptr, indices, data, N)
    n_edges = len(indices)
    density = n_edges / (N * N) * 100
    ax.imshow(mat != 0, cmap="Blues", aspect="equal", interpolation="nearest")
    ax.set_title(f"{name}\n{n_edges} edges ({density:.1f}%)")
    ax.set_xlabel("Target")
    ax.set_ylabel("Source")

plt.tight_layout()
plt.show()

## 2. Degree Distributions

Network structure is characterised by the in-degree distribution.
Random: Poisson. Scale-free: power-law tail. Ring/Grid: delta peak.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for ax, (name, (indptr, indices, data)) in zip(axes.flat, topologies.items()):
    mat = csr_to_dense(indptr, indices, data, N)
    in_deg = (mat != 0).sum(axis=0)
    out_deg = (mat != 0).sum(axis=1)
    ax.hist(in_deg, bins=20, alpha=0.6, label=f"in (mean={np.mean(in_deg):.1f})")
    ax.hist(out_deg, bins=20, alpha=0.6, label=f"out (mean={np.mean(out_deg):.1f})")
    ax.set_title(name)
    ax.set_xlabel("Degree")
    ax.set_ylabel("Count")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 3. Spiking Dynamics Under Identical Drive

We run a 100-neuron excitatory population with each topology
under the same Poisson input (80 Hz, weight 2.0) for 500 ms.
Recurrent excitation amplifies or suppresses activity depending
on connectivity structure.

In [ ]:
from sc_neurocore import StochasticLIFNeuron

results = {}
duration = 0.5  # 500 ms
dt = 0.001

for topo_name, (indptr, indices, data) in topologies.items():
    pop = Population(StochasticLIFNeuron, n=N, label="exc")

    proj = Projection(pop, pop, weight=W, topology="random", probability=0.1, seed=99)
    # Override connectivity with our topology
    proj.indptr = indptr
    proj.indices = indices
    proj.data = data.copy()

    drive = PoissonInput(n=N, rate_hz=80.0, weight=2.0, dt=dt, seed=42)
    mon = SpikeMonitor(pop, label="spk")

    net = Network(pop, proj, drive, mon)
    net.run(duration=duration, dt=dt)

    results[topo_name] = {
        "spike_trains": mon.spike_trains,
        "count": mon.count,
        "rate": mon.count / (duration * N),
    }
    print(f"{topo_name:<30s}  {mon.count:5d} spikes  {mon.count / (duration * N):6.1f} Hz")

## 4. Raster Plots

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)

for ax, (name, res) in zip(axes.flat, results.items()):
    trains = res["spike_trains"]
    for uid, times in trains.items():
        ax.scatter(np.array(times) * 1000, [uid] * len(times),
                   s=0.3, c="black", alpha=0.5)
    ax.set_ylabel("Neuron")
    ax.set_title(f"{name} — {res['count']} spikes ({res['rate']:.1f} Hz)")

axes[-1, 0].set_xlabel("Time (ms)")
axes[-1, 1].set_xlabel("Time (ms)")
plt.tight_layout()
plt.show()

## 5. Population Rate Comparison

In [ ]:
bin_width = 0.01  # 10 ms
n_bins = int(duration / bin_width)

fig, ax = plt.subplots(figsize=(12, 5))

for name, res in results.items():
    rate = np.zeros(n_bins)
    for times in res["spike_trains"].values():
        for t in times:
            b = min(int(t / bin_width), n_bins - 1)
            rate[b] += 1
    rate = rate / (N * bin_width)
    t_bins = np.arange(n_bins) * bin_width * 1000
    ax.plot(t_bins, rate, linewidth=0.8, alpha=0.8, label=name)

ax.set_xlabel("Time (ms)")
ax.set_ylabel("Population rate (Hz)")
ax.set_title("Population firing rate by topology (10 ms bins)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Summary

| Topology | Generator | Degree distribution | Typical dynamics |
|----------|-----------|--------------------|-----------------|
| Random | `random_connectivity(n, n, p, w)` | Poisson | Asynchronous irregular |
| Small-world | `small_world(n, k, p_rewire, w)` | Narrow + shortcuts | Clustered bursts |
| Scale-free | `scale_free(n, m, w)` | Power-law tail | Hub-driven cascades |
| Ring | `ring_topology(n, k, w)` | Delta (2k) | Travelling waves |
| Grid | `grid_topology(r, c, radius, w)` | Near-delta | Local patches |
| All-to-all | `all_to_all(n_src, n_tgt, w)` | Delta (N) | Synchronous |

All topologies return CSR `(indptr, indices, data)` compatible with
the `Projection` class and the Rust `NetworkRunner` backend.